# 📊 EDA - Exploratory Data Analysis for Stance Classification
This notebook computes new features and visualizes **all numeric columns** grouped by stance.

In [ ]:
# 📦 Setup
import pandas as pd
import re
import emoji
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# Load dataset
df = pd.read_csv("stance_classification_dataset_final - stance_classification_dataset_final.csv")

# Reduce to useful columns
df = df[["content", "stance", "stance_name.1"]].dropna()
df = df.rename(columns={"stance": "label", "stance_name.1": "stance_name"})

# Create plot directory
Path("eda_plots").mkdir(parents=True, exist_ok=True)

In [ ]:
# ➕ Add calculated features
def count_emojis(text):
    return sum(1 for char in text if char in emoji.EMOJI_DATA)
def has_link(text):
    return 1 if re.search(r"http[s]?://", text) else 0
def count_words(text):
    return len(text.split())
def count_uppercase(text):
    return sum(1 for c in text if c.isupper())

df["EMOJI_COUNT"] = df["content"].apply(count_emojis)
df["LINK present"] = df["content"].apply(has_link)
df["NUM_WORDS"] = df["content"].apply(count_words)
df["NUM_UPPERCASE"] = df["content"].apply(count_uppercase)

# Save enriched version
df.to_csv("cleaned_stance_dataset_enriched.csv", index=False)

In [ ]:
# 📊 Plot histograms for all numeric columns grouped by label
numeric_fields = df.select_dtypes(include=["int64", "float64"]).drop(columns=["label"], errors='ignore').columns.tolist()

for feature in numeric_fields:
    plt.figure(figsize=(8, 4))
    sns.histplot(data=df, x=feature, hue="label", bins=10, multiple="stack", palette="Set2")
    plt.title(f"{feature} distribution by stance")
    plt.xlabel(feature)
    plt.ylabel("Count")
    plt.grid(True)
    plt.tight_layout()
    plt.savefig(f"eda_plots/{feature}_by_stance.png")
    plt.close()